# Data Cleaning

## CRISP-DM Phase 3: Data Preparation

### Project: Customer Churn Analysis

Data cleaning is the process of identifying and correcting data quality issues before performing exploratory analysis and machine learning.

The objectives of this notebooks are to:

- Load the raw customer churn dataset
- Inspect the dataset before cleaning
- Standardize column names
- Identify and handle missing values
- Identify and remove dulicate records
- Correct inappropriate data types
- Check categorical values for errors
- Identify potential outliers
- Validate the cleaned dataset
- Save the cleaned dataset for downstream analysis

### Input
Raw customer churn dataset

### Output
Cleaned customer churn dataset suitable for Exploratory Data Analysis and subsequent modeling.

In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Database connection
import os
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from config.database import DB_CONFIG
from sqlalchemy import create_engine

# Display settings
pd.set_option("display.max_columns", None)

# Plot settings
plt.style.use("ggplot")

print("Libraries imported successfully.")

Libraries imported successfully.


In [47]:
# Processed Data Path
PROCESSED_DATA_PATH = Path("data/processed/customer_churn_clean.csv")
print(PROCESSED_DATA_PATH)

data/processed/customer_churn_clean.csv


## 1. Load Raw Dataset

The raw dataset is loaded withoput modifying the original state.

Keeping the raw dataset unchanged is important because it provides a reliable source that can be revisited if a preprocessing decision needs to be changed.

In [4]:
# Load dataset
engine = create_engine(
    f"postgresql://{DB_CONFIG["user"]}:{DB_CONFIG["password"]}@{DB_CONFIG["host"]}:{DB_CONFIG["port"]}/{DB_CONFIG["database"]}"
)

query = "SELECT * FROM customer_churn;"
df = pd.read_sql(query, engine)
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [5]:
df_clean = df.copy()
print("Working copy created")

Working copy created


## 2. Initial Data Inspection

Before making any changes, the dataset is inspected to establish a baseline.

This allows us to compare the dataset before and after cleaning

In [6]:
df_clean.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [7]:
df_clean.tail()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
7038,2569-WGERO,1,United States,California,Landers,92285,"34.341737, -116.539416",34.341737,-116.539416,Female,No,No,No,72,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),21.15,1419.4,No,0,45,5306,NaN
7039,6840-RESVB,1,United States,California,Adelanto,92301,"34.667815, -117.536183",34.667815,-117.536183,Male,No,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No,0,59,2140,NaN
7040,2234-XADUH,1,United States,California,Amboy,92304,"34.559882, -115.637164",34.559882,-115.637164,Female,No,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No,0,71,5560,NaN
7041,4801-JZAZL,1,United States,California,Angelus Oaks,92305,"34.1678, -116.86433",34.167800,-116.864330,Female,No,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No,0,59,2793,NaN
7042,3186-AJIEK,1,United States,California,Apple Valley,92308,"34.424926, -117.184503",34.424926,-117.184503,Male,No,No,No,66,Yes,No,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No,0,38,5097,NaN


In [8]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Service   7043 non-null   

In [9]:
baseline = pd.DataFrame({
    "Column": df_clean.columns,
    "Data Types": df_clean.dtypes.astype(str).values,
    "Missing Data":df_clean.isna().sum().values,
    "Missing %": (df_clean.isna().mean()*100).round(2).values,
    "Unique Values": df_clean.nunique().values
})
baseline

,Column,Data Types,Missing Data,Missing %,Unique Values
0,CustomerID,str,0,0.00,7043
1,Count,int64,0,0.00,1
2,Country,str,0,0.00,1
3,State,str,0,0.00,1
4,City,str,0,0.00,1129
5,Zip Code,int64,0,0.00,1652
6,Lat Long,str,0,0.00,1652
7,Latitude,float64,0,0.00,1652
8,Longitude,float64,0,0.00,1651
9,Gender,str,0,0.00,2


## 3. Standarize Column Names

Column names are standardized to make them easier to use in python and SQL queries.

The folling conventions will be applied:

- Convert names to lowercase
- Remove leading and trailing spaces
- Replace spaces with underscore

In [10]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
df_clean.columns.to_list()

['customerid',
 'count',
 'country',
 'state',
 'city',
 'zip_code',
 'lat_long',
 'latitude',
 'longitude',
 'gender',
 'senior_citizen',
 'partner',
 'dependents',
 'tenure_months',
 'phone_service',
 'multiple_lines',
 'internet_service',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'contract',
 'paperless_billing',
 'payment_method',
 'monthly_charges',
 'total_charges',
 'churn_label',
 'churn_value',
 'churn_score',
 'cltv',
 'churn_reason']

## 4. Duplicate Records
Duplicate records can distort statistical analysis and machine learning models.

We will identify duplicate rows before deciding whether they should be removed.

In [11]:
duplicate_count = df_clean.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


In [12]:
df_clean = df_clean.drop_duplicates()
print(f"Rows after removing duplicated: {len(df_clean)}")

Rows after removing duplicated: 7043


In [13]:
# Checking for duplicated in the Customer ID
duplicate_customers = (
    df_clean["customerid"]
    .duplicated()
    .sum()
)

print(f"Duplicate Customer IDs: {duplicate_customers}")

Duplicate Customer IDs: 0


## 5. Missing Value Analysis

Missing values must be investigated before deciding how to handle them.

We will examine:

- Number of missing values
- Percentage of missing values
- Columns affected

In [14]:
missing_report = pd.DataFrame({
    "Missing Values": df_clean.isna().sum(),
    "Missing Percentage": (
        df_clean.isna().mean() * 100
    ).round(2)
})

missing_report = (
    missing_report
    .sort_values("Missing Values", ascending=False)
)

missing_report

,Missing Values,Missing Percentage
churn_reason,5174,73.46
customerid,0,0.00
count,0,0.00
state,0,0.00
country,0,0.00
zip_code,0,0.00
lat_long,0,0.00
latitude,0,0.00
city,0,0.00
gender,0,0.00


In [15]:
# Examine churn reason by churn value

churn_reason_analysis = pd.crosstab(
    df_clean['churn_label'],
    df_clean['churn_reason'].isna(),
    margins=True
)
churn_reason_analysis

churn_reason,False,True,All
churn_label,,,
No,0,5174,5174
Yes,1869,0,1869
All,1869,5174,7043


In [16]:
# Display missing churn reasons among customers who churned

df_clean[
    (df_clean["churn_label"] == "Yes") &
    (df_clean["churn_reason"].isna())
][[
    "customerid",
    "churn_label",
    "churn_reason"
]]

,customerid,churn_label,churn_reason


In [17]:
# Replace missing churn reasons for customers who did not churn

df_clean.loc[
    df_clean["churn_label"] == "No",
    "churn_reason"
] = "Not Applicable"

In [18]:
# Verify the transformation worked
print(
    "Missing churn_reason values:",
    df_clean["churn_reason"].isna().sum()
)

Missing churn_reason values: 0


### Handling Missing 'churn_reason'

The 'churn_reason' column initally contained 5, 174 missing values.
Investigation shoed that all missing values belonged to the customers with `churn_label = No`. Since these customers did not churn, a churn reason is not applicable.

Therefor, the missing values were replaced with `"Not Applicable"` to explicitly represent the business meanig of the missing value.

All customers with `churn_label = Yes` already have a recorded churn reason, so no imputation was required for churned customers.

This transformation converts structurally missing values into an explicit business category without introducing assumptions about the customer's reason for not churning.

## 6. Identifying and Handling Inconsistent Values

Analyzed dataset columns to identify inconsistent, invalid, or unexpected values.

Applied data-cleaning techniques to improve data quality and ensure consistency for further analysis.

In [19]:
# Check categorical values
categorical_columns = df_clean.select_dtypes(
    include="object"
).columns

for column in categorical_columns:
    print(f"\n{column}")
    print(df_clean[column].unique())


customerid
<StringArray>
['3668-QPYBK', '9237-HQITU', '9305-CDSKC', '7892-POOKP', '0280-XJGEX',
 '4190-MFLUW', '8779-QRDMV', '1066-JKSGK', '6467-CHFZW', '8665-UTDHZ',
 ...
 '0871-OPBXW', '3605-JISKB', '9767-FFLEM', '8456-QDAVC', '7750-EYXWZ',
 '2569-WGERO', '6840-RESVB', '2234-XADUH', '4801-JZAZL', '3186-AJIEK']
Length: 7043, dtype: str

country
<StringArray>
['United States']
Length: 1, dtype: str

state
<StringArray>
['California']
Length: 1, dtype: str

city
<StringArray>
[    'Los Angeles',   'Beverly Hills', 'Huntington Park',         'Lynwood',
  'Marina Del Rey',       'Inglewood',    'Santa Monica',        'Torrance',
        'Whittier',        'La Habra',
 ...
      'Janesville',      'Litchfield',        'Loyalton',        'Madeline',
    'Markleeville',         'Milford',         'Calpine',        'Standish',
        'Tulelake',  'Olympic Valley']
Length: 1129, dtype: str

lat_long
<StringArray>
['33.964131, -118.272783',  '34.059281, -118.30742', '34.048013, -118.293953',


<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

online_security
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

online_backup
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

device_protection
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

tech_support
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

streaming_tv
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

streaming_movies
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

contract
<StringArray>
['Month-to-month', 'Two year', 'One year']
Length: 3, dtype: str

paperless_billing
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

payment_method
<StringArray>
[             'Mailed check',          'Electronic check',
 'Bank transfer (automatic)',   'Credit card (automatic)']
Length: 4, dtype: str

total_charges
<StringArray>
[ '108.15',  '151.65',   '820.5', '3046.05'

/tmp/ipykernel_5618/3803508932.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df_clean.select_dtypes(


In [20]:
# Check Categorical Distribution
for columns in categorical_columns:
    print(f"\n--- {column} ---")
    print(df_clean[column].value_counts(dropna=False))


--- churn_reason ---
churn_reason
Not Applicable                               5174
Attitude of support person                    192
Competitor offered higher download speeds     189
Competitor offered more data                  162
Don't know                                    154
Competitor made better offer                  140
Attitude of service provider                  135
Competitor had better devices                 130
Network reliability                           103
Product dissatisfaction                       102
Price too high                                 98
Service dissatisfaction                        89
Lack of self-service on Website                88
Extra data charges                             57
Moved                                          53
Limited range of services                      44
Lack of affordable download/upload speed       44
Long distance charges                          44
Poor expertise of phone support                20
Poor expertise 

In [21]:
# Standardizing categorical text
for column in categorical_columns:
    df_clean[column] = df_clean[column].str.strip()

for column in categorical_columns:
    print(f"{column}: {df_clean[column].unique()}")

customerid: <StringArray>
['3668-QPYBK', '9237-HQITU', '9305-CDSKC', '7892-POOKP', '0280-XJGEX',
 '4190-MFLUW', '8779-QRDMV', '1066-JKSGK', '6467-CHFZW', '8665-UTDHZ',
 ...
 '0871-OPBXW', '3605-JISKB', '9767-FFLEM', '8456-QDAVC', '7750-EYXWZ',
 '2569-WGERO', '6840-RESVB', '2234-XADUH', '4801-JZAZL', '3186-AJIEK']
Length: 7043, dtype: str
country: <StringArray>
['United States']
Length: 1, dtype: str
state: <StringArray>
['California']
Length: 1, dtype: str
city: <StringArray>
[    'Los Angeles',   'Beverly Hills', 'Huntington Park',         'Lynwood',
  'Marina Del Rey',       'Inglewood',    'Santa Monica',        'Torrance',
        'Whittier',        'La Habra',
 ...
      'Janesville',      'Litchfield',        'Loyalton',        'Madeline',
    'Markleeville',         'Milford',         'Calpine',        'Standish',
        'Tulelake',  'Olympic Valley']
Length: 1129, dtype: str
lat_long: <StringArray>
['33.964131, -118.272783',  '34.059281, -118.30742', '34.048013, -118.293953',


In [22]:
df_clean["gender"].value_counts()

gender
Male      3555
Female    3488
Name: count, dtype: int64

In [23]:
df_clean["partner"].value_counts()

partner
No     3641
Yes    3402
Name: count, dtype: int64

In [24]:
df_clean["dependents"].value_counts()

dependents
No     5416
Yes    1627
Name: count, dtype: int64

In [27]:
df_clean["churn_label"].value_counts()

churn_label
No     5174
Yes    1869
Name: count, dtype: int64

In [28]:
# Checking numberic data
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
count,7043.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
zip_code,7043.0,93521.964646,1865.794555,90001.000000,92102.000000,93552.000000,95351.000000,96161.000000
latitude,7043.0,36.282441,2.455723,32.555828,34.030915,36.391777,38.224869,41.962127
longitude,7043.0,-119.798880,2.157889,-124.301372,-121.815412,-119.730885,-118.043237,-114.192901
tenure_months,7043.0,32.371149,24.559481,0.000000,9.000000,29.000000,55.000000,72.000000
monthly_charges,7043.0,64.761692,30.090047,18.250000,35.500000,70.350000,89.850000,118.750000
churn_value,7043.0,0.265370,0.441561,0.000000,0.000000,0.000000,1.000000,1.000000
churn_score,7043.0,58.699418,21.525131,5.000000,40.000000,61.000000,75.000000,100.000000
cltv,7043.0,4400.295755,1183.057152,2003.000000,3469.000000,4527.000000,5380.500000,6500.000000


In [ ]:
# Looking for impossible values
df_clean["tenure_months"].min(), df_clean["tenure_months"].max()

(np.int64(0), np.int64(72))

In [30]:
df_clean["monthly_charges"].min(), df_clean["monthly_charges"].max()

(np.float64(18.25), np.float64(118.75))

In [31]:
df_clean["total_charges"].min(), df_clean["total_charges"].max()

('', '999.9')

In [37]:
# The "total_charges" column is in the string datatype which needs to be converted to float.
df_clean["total_charges"] = (
    df_clean["total_charges"]
    .replace(r"^\s*$", 0, regex=True)
    .astype(float)
)

In [38]:
# Business value validation
assert df_clean["tenure_months"].min() >= 0
assert df_clean["monthly_charges"].min() >= 0
assert df_clean["total_charges"].min() >= 0

In [40]:
# Checking Logical relationship
df_clean[
    (df_clean["tenure_months"] > 0) &
    (df_clean["total_charges"] == 0)
][
    ["customerid", "tenure_months", "monthly_charges", "total_charges"]
]

,customerid,tenure_months,monthly_charges,total_charges


## 7. Outlier Investigation
Identified unusual values that deviate significantly from the expected data range.
Investigated potential outliers to determine whether they represent genuine observations or data quality issues.


In [41]:
numerical_columns = df_clean.select_dtypes(
    include=np.number
).columns

df_clean[numerical_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
count,7043.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
zip_code,7043.0,93521.964646,1865.794555,90001.000000,92102.000000,93552.000000,95351.000000,96161.000000
latitude,7043.0,36.282441,2.455723,32.555828,34.030915,36.391777,38.224869,41.962127
longitude,7043.0,-119.798880,2.157889,-124.301372,-121.815412,-119.730885,-118.043237,-114.192901
tenure_months,7043.0,32.371149,24.559481,0.000000,9.000000,29.000000,55.000000,72.000000
monthly_charges,7043.0,64.761692,30.090047,18.250000,35.500000,70.350000,89.850000,118.750000
total_charges,7043.0,2279.734304,2266.794470,0.000000,398.550000,1394.550000,3786.600000,8684.800000
churn_value,7043.0,0.265370,0.441561,0.000000,0.000000,0.000000,1.000000,1.000000
churn_score,7043.0,58.699418,21.525131,5.000000,40.000000,61.000000,75.000000,100.000000
cltv,7043.0,4400.295755,1183.057152,2003.000000,3469.000000,4527.000000,5380.500000,6500.000000


In [43]:
# Checking for values less than 0
df_clean[
    (df_clean["monthly_charges"] < 0) |
    (df_clean["total_charges"] < 0) |
    (df_clean["tenure_months"] < 0)
]

,customerid,count,country,state,city,zip_code,lat_long,latitude,longitude,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason


In [44]:
# Final Data Quality Check
final_quality_report = pd.DataFrame({
    "Data Type": df_clean.dtypes.astype(str),
    "Missing Values": df_clean.isna().sum(),
    "Missing %": (df_clean.isna().mean() * 100).round(2),
    "Unique Values": df_clean.nunique()
})

final_quality_report

,Data Type,Missing Values,Missing %,Unique Values
customerid,str,0,0.0,7043
count,int64,0,0.0,1
country,str,0,0.0,1
state,str,0,0.0,1
city,str,0,0.0,1129
zip_code,int64,0,0.0,1652
lat_long,str,0,0.0,1652
latitude,float64,0,0.0,1652
longitude,float64,0,0.0,1651
gender,str,0,0.0,2


In [45]:
# Comparing Before and After Data Cleaning

comparison = pd.DataFrame({
    "Metric":[
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Missing Values"
    ],
    "Before Cleaning":[
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        df.isna().sum().sum()
    ],
    "After Cleanin":[
        df_clean.shape[0],
        df_clean.shape[1],
        df_clean.duplicated().sum(),
        df_clean.isna().sum().sum()
    ]
})
comparison

,Metric,Before Cleaning,After Cleanin
0,Rows,7043,7043
1,Columns,33,33
2,Duplicate Rows,0,0
3,Missing Values,5174,0


## 8. Saving Cleaned Dataset
Saved the cleaned and validated dataset for further analysis and modeling.
Ensured the final dataset has consistent formats, handled missing values, and improved overall data quality.

In [49]:
PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
# Saving the cleaned dataset to .csv file
df_clean.to_csv(
    PROCESSED_DATA_PATH,
    index=False
)
print("Cleaned dataset saved successfully!")
print(PROCESSED_DATA_PATH)

Cleaned dataset saved successfully!
data/processed/customer_churn_clean.csv


In [52]:
# Saving the cleaned dataset to Postgresql database
df_clean.to_sql(
    "customer_churn_clean",
    engine,
    if_exists="replace",
    index=False
)

113

# Cnclusion

The raw customer churn dataset was successfully cleaned and validated.

### Cleaning activities performed

- Loaded the raw dataset without modifying the original source.
- Created a working copy for preprocessing.
- Standardized column names.
- Checked for duplicate records.
- Checked for duplicate customer IDs.
- Identified missing values.
- Converted `total_charges` to a numeric data type.
- Investigated missing values in `churn_reason`.
- Checked categorical values for inconsistencies.
- Validated numerical ranges.
- Applied business-rule checks.
- Performed an initial outlier investigation.
- Validated the final dataset.
- Saved the cleaned dataset to the processed data directory and Postgresql database.

The cleaned dataset will be used as the input for the Exploratory Data Analysis phase.